In [30]:
!pip -q install pymongo google-genai pandas

In [31]:
import time
import json
import datetime
import pandas as pd

from google.colab import userdata
from google import genai
from pymongo import MongoClient

MODEL = "gemini-3.1-flash-lite"

client = genai.Client(
    api_key=userdata.get("GEMINI_API_KEY"),
    http_options={"timeout": 30000}
)

mongo_client = MongoClient(
    userdata.get("MONGODB_URI"),
    serverSelectionTimeoutMS=8000
)

mongo_client.admin.command("ping")

db = mongo_client["genai_course"]

print("Gemini connected successfully!")
print("MongoDB connected successfully!")

OperationFailure: bad auth : authentication failed, full error: {'ok': 0, 'errmsg': 'bad auth : authentication failed', 'code': 8000, 'codeName': 'AtlasError'}

In [32]:
from google.colab import userdata
from urllib.parse import urlparse

uri = userdata.get("MONGODB_URI").strip()

parsed = urlparse(uri)

print("Protocol:", parsed.scheme)
print("Hostname:", parsed.hostname)
print("Username present:", bool(parsed.username))
print("Password present:", bool(parsed.password))
print("Valid MongoDB hostname:",
      bool(parsed.hostname and parsed.hostname.endswith(".mongodb.net")))

Protocol: mongodb+srv
Hostname: cluster1.imjtmfw.mongodb.net
Username present: True
Password present: True
Valid MongoDB hostname: True


In [33]:
from google.colab import userdata

uri = userdata.get("MONGODB_URI").strip()

print("Secret length:", len(uri))
print("Starts correctly:", uri.startswith("mongodb+srv://"))
print("Ends correctly:", uri.endswith("majority"))

Secret length: 90
Starts correctly: True
Ends correctly: False


In [34]:
from pymongo import MongoClient

mongo_client = MongoClient(
    uri,
    serverSelectionTimeoutMS=15000
)

try:
    print(mongo_client.admin.command("ping"))
    print("MongoDB connected successfully!")

    db = mongo_client["genai_course"]

except Exception as e:
    print("Connection failed:", type(e).__name__)
    print(str(e)[:500])

Connection failed: OperationFailure
bad auth : authentication failed, full error: {'ok': 0, 'errmsg': 'bad auth : authentication failed', 'code': 8000, 'codeName': 'AtlasError'}


In [35]:
from google.colab import userdata
from urllib.parse import urlparse

uri = userdata.get("MONGODB_URI").strip()
parsed = urlparse(uri)

print("Hostname:", parsed.hostname)
print("Username:", parsed.username)
print("Password exists:", bool(parsed.password))
print("Secret length:", len(uri))

Hostname: cluster1.imjtmfw.mongodb.net
Username: vaishnavisubravet97_db_user
Password exists: True
Secret length: 90


In [36]:
from google.colab import userdata

try:
    password = userdata.get("MONGO_PASSWORD")
    print("MONGO_PASSWORD secret found:", bool(password))
    print("Password length:", len(password) if password else 0)
except Exception as e:
    print("Secret error:", type(e).__name__)

MONGO_PASSWORD secret found: True
Password length: 18


In [37]:
from pymongo import MongoClient
from urllib.parse import quote_plus
from google.colab import userdata

# MongoDB credentials
username = "vaishnavisubravet97_db_user"
password = userdata.get("MONGO_PASSWORD")

# Your actual Atlas cluster
host = "cluster1.imjtmfw.mongodb.net"

# Build connection string safely
uri = (
    f"mongodb+srv://{quote_plus(username)}:"
    f"{quote_plus(password)}@{host}/"
    "?authSource=admin&retryWrites=true&w=majority"
)

# Connect
client = MongoClient(uri, serverSelectionTimeoutMS=20000)

try:
    result = client.admin.command("ping")

    print("MongoDB connected successfully!")
    print(result)

    db = client["genai_course"]

except Exception as e:
    print("Connection failed:", type(e).__name__)
    print(str(e)[:300])

Connection failed: OperationFailure
bad auth : authentication failed, full error: {'ok': 0, 'errmsg': 'bad auth : authentication failed', 'code': 8000, 'codeName': 'AtlasError'}


In [38]:
from google.colab import userdata
from pymongo import MongoClient
from urllib.parse import quote_plus

username = "vaishnavisubravet97_db_user"
password = userdata.get("MONGO_PASSWORD").strip()

host = "cluster1.imjtmfw.mongodb.net"

uri = (
    f"mongodb+srv://{quote_plus(username)}:"
    f"{quote_plus(password)}@{host}/"
    "?retryWrites=true&w=majority&authSource=admin"
)

client = MongoClient(uri, serverSelectionTimeoutMS=20000)

try:
    print(client.admin.command("ping"))
    print("SUCCESS! MongoDB is connected.")

    db = client["genai_course"]

except Exception as e:
    print(type(e).__name__)
    print(str(e)[:300])

{'ok': 1}
SUCCESS! MongoDB is connected.


In [39]:
import json

# Check that Golden Question Set exists
print("Total golden questions:", len(golden))

# Save questions into MongoDB
for question in golden:
    db.golden_set.replace_one(
        {"id": question["id"]},
        question,
        upsert=True
    )

# Save a local JSON backup
with open("golden_set.json", "w") as f:
    json.dump(golden, f, indent=2)

print("Golden Question Set saved successfully!")

# Verify MongoDB
print("Questions stored in MongoDB:",
      db.golden_set.count_documents({}))

Total golden questions: 32
Golden Question Set saved successfully!
Questions stored in MongoDB: 32


In [40]:
import json

# Create Golden Question Set
GOLDEN = [
    ("What does BPE stand for?", ["byte", "pair"], "Module I"),
    ("Which tokenizer does BERT use?", ["wordpiece"], "Module I"),
    ("Which tokenizer family do T5 and mT5 use?", ["sentencepiece"], "Module I"),
    ("Why does Telugu text usually cost more tokens than English?", ["token"], "Module I"),
    ("What is a knowledge cutoff of an LLM?", ["date"], "Module I"),
    ("What is a hallucination in an LLM?", ["false"], "Module I"),
    ("What is next-token prediction?", ["next", "token"], "Module I"),
    ("What is a context window?", ["token"], "Module I"),

    ("Name the three vectors used in attention.", ["query", "key", "value"], "Module II"),
    ("Which function turns attention scores into weights?", ["softmax"], "Module II"),
    ("Is GPT encoder-only or decoder-only?", ["decoder"], "Module II"),
    ("Is BERT encoder-only or decoder-only?", ["encoder"], "Module II"),
    ("What kind of architecture is T5?", ["encoder", "decoder"], "Module II"),
    ("What does a higher temperature do during sampling?", ["random"], "Module II"),
    ("What is top-k sampling?", ["k"], "Module II"),
    ("What is top-p sampling also called?", ["nucleus"], "Module II"),

    ("What does TF-IDF stand for?", ["term frequency", "inverse document frequency"], "Module III"),
    ("Name a classic sparse ranking function used in search engines.", ["bm25"], "Module III"),
    ("What does HNSW stand for?", ["hierarchical", "navigable", "small world"], "Module III"),
    ("What does RAG stand for?", ["retrieval", "augmented", "generation"], "Module III"),
    ("What does MRR stand for in retrieval evaluation?", ["mean", "reciprocal", "rank"], "Module III"),
    ("Why do we add overlap when chunking documents?", ["context"], "Module III"),

    ("What is few-shot prompting?", ["example"], "Module IV"),
    ("Which Python library is used for schema-validated structured outputs?", ["pydantic"], "Module IV"),
    ("What is prompt injection?", ["instruction"], "Module IV"),

    ("What does LoRA stand for?", ["low", "rank", "adaptation"], "Module V"),
    ("What does the Q in QLoRA refer to?", ["quantiz"], "Module V"),
    ("What does DPO stand for?", ["direct", "preference", "optimization"], "Module V"),
    ("What does RLHF stand for?", ["reinforcement", "human", "feedback"], "Module V"),

    ("Which Python framework is used for microservices?", ["fastapi"], "Module VI"),
    ("Which tool is used for Infrastructure as Code on AWS?", ["terraform"], "Module VI"),
    ("What does MCP stand for?", ["model", "context", "protocol"], "Module VI")
]

golden = [
    {
        "id": f"Q{i+1:02d}",
        "question": q,
        "expected_keywords": keywords,
        "source": module
    }
    for i, (q, keywords, module) in enumerate(GOLDEN)
]

# Save JSON backup
with open("golden_set.json", "w") as f:
    json.dump(golden, f, indent=2)

# Save to MongoDB
for question in golden:
    db.golden_set.replace_one(
        {"id": question["id"]},
        question,
        upsert=True
    )

print("Golden Question Set created successfully!")
print("Total questions:", len(golden))
print("MongoDB records:", db.golden_set.count_documents({}))

Golden Question Set created successfully!
Total questions: 32
MongoDB records: 32


In [41]:
!pip install -q "google-genai>=2.12.1,<3" "google-auth==2.49.0" pymongo pandas

In [42]:
from google.colab import userdata
from google import genai
from pymongo import MongoClient
from urllib.parse import quote_plus

# Gemini connection
MODEL = "gemini-3.1-flash-lite"

client = genai.Client(
    api_key=userdata.get("GEMINI_API_KEY")
)

# MongoDB connection
username = "vaishnavisubravet97_db_user"
password = userdata.get("MONGO_PASSWORD").strip()
host = "cluster1.imjtmfw.mongodb.net"

uri = (
    f"mongodb+srv://{quote_plus(username)}:"
    f"{quote_plus(password)}@{host}/"
    "?retryWrites=true&w=majority"
)

mongo_client = MongoClient(uri)
mongo_client.admin.command("ping")

db = mongo_client["genai_course"]

print("Gemini and MongoDB connections initialized!")

Gemini and MongoDB connections initialized!


In [43]:
print("Golden questions in MongoDB:",
      db.golden_set.count_documents({}))

Golden questions in MongoDB: 32


In [44]:
from google.colab import userdata
from google import genai
from pymongo import MongoClient
from urllib.parse import quote_plus

MODEL = "gemini-3.1-flash-lite"

client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))

username = "vaishnavisubravet97_db_user"
password = userdata.get("MONGO_PASSWORD").strip()
host = "cluster1.imjtmfw.mongodb.net"

uri = (
    f"mongodb+srv://{quote_plus(username)}:"
    f"{quote_plus(password)}@{host}/"
    "?retryWrites=true&w=majority"
)

mongo_client = MongoClient(uri, serverSelectionTimeoutMS=20000)
mongo_client.admin.command("ping")

db = mongo_client["genai_course"]
golden = list(db.golden_set.find({}, {"_id": 0}))

print("Gemini connected!")
print("MongoDB connected!")
print("Golden questions:", len(golden))

Gemini connected!
MongoDB connected!
Golden questions: 32


In [45]:
import time
import pandas as pd

def ask(prompt, retries=4):
    for attempt in range(retries):
        try:
            return client.models.generate_content(
                model=MODEL,
                contents=prompt
            )
        except Exception as e:
            print(f"Retry {attempt + 1}: {str(e)[:150]}")
            time.sleep(5 * (attempt + 1))

    raise RuntimeError("Gemini request failed after 4 attempts.")

def score(answer, keywords):
    answer = answer.lower().replace("-", " ")
    return all(keyword.lower() in answer for keyword in keywords)

results = []

for i, g in enumerate(golden):
    start = time.time()

    response = ask(
        "Answer in one or two sentences: " + g["question"]
    )

    answer = (response.text or "").strip()
    usage = response.usage_metadata

    results.append({
        "id": g["id"],
        "source": g["source"],
        "correct": score(answer, g["expected_keywords"]),
        "latency_s": round(time.time() - start, 2),
        "input_tokens": usage.prompt_token_count if usage else 0,
        "output_tokens": usage.candidates_token_count if usage else 0,
        "answer": answer[:150]
    })

    print(f"Completed {i + 1}/32")

    time.sleep(4)

df = pd.DataFrame(results)

print("\nBASELINE TEST COMPLETED!")
display(df)

Completed 1/32
Completed 2/32
Completed 3/32
Completed 4/32
Completed 5/32
Completed 6/32
Completed 7/32
Completed 8/32
Completed 9/32
Completed 10/32
Completed 11/32
Completed 12/32
Completed 13/32
Completed 14/32
Completed 15/32
Completed 16/32
Completed 17/32
Completed 18/32
Completed 19/32
Completed 20/32
Completed 21/32
Completed 22/32
Completed 23/32
Completed 24/32
Completed 25/32
Completed 26/32
Completed 27/32
Completed 28/32
Completed 29/32
Completed 30/32
Completed 31/32
Completed 32/32

BASELINE TEST COMPLETED!


,id,source,correct,latency_s,input_tokens,output_tokens,answer
0,Q01,Module I,True,0.96,15,33,BPE stands for Byte Pair Encoding. It is a sub...
1,Q02,Module I,True,2.92,14,29,"BERT uses the **WordPiece** tokenizer, which b..."
2,Q03,Module I,True,2.63,19,44,T5 and mT5 use the **SentencePiece** tokenizer...
3,Q04,Module I,True,2.38,19,69,Telugu uses the Unicode-based UTF-8 encoding w...
4,Q05,Module I,True,1.57,18,53,A knowledge cutoff is the date marking the end...
5,Q06,Module I,False,3.55,18,57,A hallucination in an LLM occurs when the mode...
6,Q07,Module I,True,2.11,15,57,Next-token prediction is a machine learning te...
7,Q08,Module I,True,2.90,14,64,A context window is the maximum amount of text...
8,Q09,Module II,True,1.95,16,25,The three vectors used in the attention mechan...
9,Q10,Module II,True,2.89,16,24,The **softmax** function is used to normalize ...


In [46]:
import datetime
import json

summary = {
    "release": "v0.1",
    "model": MODEL,
    "date": str(datetime.date.today()),
    "questions": len(df),
    "passed": int(df["correct"].sum()),
    "failed": int((~df["correct"]).sum()),
    "accuracy": round(df["correct"].mean() * 100, 2),
    "avg_latency_s": round(df["latency_s"].mean(), 2),
    "avg_output_tokens": round(df["output_tokens"].mean(), 2)
}

# Save baseline results
df.to_csv("baseline_v0.1.csv", index=False)

# Save summary as JSON
with open("baseline_v0.1_report.json", "w") as f:
    json.dump(summary, f, indent=4)

# Store release information in MongoDB
db.releases.update_one(
    {"release": "v0.1"},
    {"$set": summary},
    upsert=True
)

print("===== RELEASE v0.1 BASELINE REPORT =====")
for key, value in summary.items():
    print(f"{key}: {value}")

print("\n===== MODULE-WISE ACCURACY =====")
print((df.groupby("source")["correct"].mean() * 100).round(2))

print("\nBaseline saved successfully!")

===== RELEASE v0.1 BASELINE REPORT =====
release: v0.1
model: gemini-3.1-flash-lite
date: 2026-11-16
questions: 32
passed: 31
failed: 1
accuracy: 96.88
avg_latency_s: 2.54
avg_output_tokens: 43.19

===== MODULE-WISE ACCURACY =====
source
Module I       87.5
Module II     100.0
Module III    100.0
Module IV     100.0
Module V      100.0
Module VI     100.0
Name: correct, dtype: float64

Baseline saved successfully!
